# E4b: desplazamiento de la ventana de la etiqueta

W = 60 min; con desplazamiento *b*, la etiqueta en *t* mira [*t* − *b*, *t* + 60 − *b*). Se prueban *b* = 0, 15 y 30. Modelos: F con toda la dinámica (E8d) y S con la referencia podada.

Criterio fijado antes (`README.md`, E4b): un *b* se adopta si suma 0.01 o más de AUC medio frente a *b* = 0 y, en cada persona, el modelo supera al más fuerte de tres baselines: persistencia de la última hora, **persistencia desplazada** (la hora anterior a la ventana desplazada, [*t* − *b* − 60, *t* − *b*), sobre el perfil) y **parte observada** (F o S de [*t* − *b*, *t*), lo que ya ocurrió de la ventana de la etiqueta).

In [1]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
from experiments.modeling import feature_cols, evaluate, persistence_score
from experiments import variants as V
from pipelines.labels.build_windows import seconds
from smartuse.features_dynamics import GROUPS
from smartuse.windows import window_metrics
SEEDS, OFFSETS, TARGETS = [0, 1, 2, 3, 4], [0, 15, 30], ["y_f", "y_s"]

def tables(name, b):
    sfx = "" if b == 0 else f"_b{b}"
    base = pd.read_csv(f"data/processed/{name}_features{sfx}.csv", parse_dates=["day"])
    dyn = pd.read_csv(f"data/processed/{name}_features_dynamics{sfx}.csv")
    assert (base["t_utc"].values == dyn["t_utc"].values).all()
    return pd.concat([base, dyn.drop(columns=["user_id", "t_utc"])], axis=1)

data = {b: (tables("real", b), tables("mpud", b)) for b in OFFSETS}
pruned = V.pruned(feature_cols(pd.read_csv("data/processed/real_features.csv", nrows=1)))
COLS = {"y_f": pruned + [c for cols in GROUPS.values() for c in cols], "y_s": pruned}
pd.DataFrame({b: {"examples": len(r), "y_f": r.y_f.mean(), "y_s": r.y_s.mean()} for b, (r, _) in data.items()}).round(3)

,0,15,30
examples,16281.000,16302.000,16318.000
y_f,0.265,0.270,0.276
y_s,0.177,0.174,0.171


## Corridas (3 desplazamientos × 2 modelos × 5 semillas)

Scores en `data/reports/e4b_scores_b{b}.parquet`, reutilizados si existen.

In [2]:
scores = {}
for b, (real, mpud) in data.items():
    cache = Path(f"data/reports/e4b_scores_b{b}.parquet")
    sc = pd.read_parquet(cache) if cache.exists() else pd.DataFrame(index=real.index)
    for t in TARGETS:
        for seed in SEEDS:
            key = f"{t}|{seed}"
            if key not in sc:
                V.set_seed(seed)
                sc[key] = V.scores_for(real, mpud, COLS[t], t)
                sc.to_parquet(cache)
    scores[b] = sc
runs = pd.DataFrame([{"b": b, "target": k.split("|")[0], "seed": int(k.split("|")[1]),
                      "auc": V.auc(data[b][0], scores[b][k].to_numpy(), k.split("|")[0])}
                     for b in OFFSETS for k in scores[b]])
mean = runs.pivot_table(index="b", columns="target", values="auc", aggfunc="mean")
pd.concat({"AUC (media)": mean, "frente a b = 0": mean - mean.loc[0],
           "mín.": runs.pivot_table(index="b", columns="target", values="auc", aggfunc="min"),
           "máx.": runs.pivot_table(index="b", columns="target", values="auc", aggfunc="max")}, axis=1).round(3)

AUC (media)        frente a b = 0          mín.          máx.       
target         y_f    y_s            y_f    y_s    y_f    y_s    y_f    y_s
b                                                                          
0            0.705  0.689          0.000  0.000  0.693  0.679  0.715  0.699
15           0.846  0.787          0.141  0.098  0.843  0.782  0.848  0.792
30           0.914  0.951          0.209  0.263  0.912  0.946  0.915  0.954

## Baselines por persona

Persistencia: F₆₀ o S₆₀ de la última hora sobre el perfil (del mismo *b*). Persistencia desplazada: la hora [*t* − *b* − 60, *t* − *b*) sobre el perfil (igual a la persistencia con *b* = 0). Parte observada: F y S de [*t* − *b*, *t*) con W = 60, como score continuo (no existe con *b* = 0).

In [3]:
dates = ["start_utc", "start_local", "end_utc"]
sessions = pd.read_csv("data/processed/real_preprocess_3.csv", parse_dates=dates)
logs = {u: (seconds(s.start_utc), seconds(s.end_utc)) for u, s in sessions.groupby("user_id")}

def past_window(real, start_back_min, length_min):
    # F and S of [t - start_back, t - start_back + length), from the log up to t (W = 60 min)
    f, s = np.zeros(len(real)), np.zeros(len(real))
    for i, (u, t) in enumerate(zip(real.user_id, seconds(pd.to_datetime(real.t_utc)))):
        st, en = logs[u]
        a = t - start_back_min * 60
        m = window_metrics(st, np.minimum(en, min(t, a + length_min * 60)), a, 3600.0)
        f[i], s[i] = m.f, m.s
    return {"y_f": f, "y_s": s}

summary = {}
for b, (real, _) in data.items():
    persist = {"y_f": (real.f_60 > real.prof_pf).astype(float).to_numpy(),
               "y_s": (real.s_60 > real.prof_ps).astype(float).to_numpy()}
    obs = past_window(real, b, b) if b else None             # [t - b, t)
    prev = past_window(real, b + 60, 60)                        # [t - b - 60, t - b)
    shifted = {"y_f": (prev["y_f"] > real.prof_pf).astype(float).to_numpy(),
               "y_s": (prev["y_s"] > real.prof_ps).astype(float).to_numpy()}
    for t in TARGETS:
        pp = lambda score: pd.Series(evaluate(real[t], score, real.user_id, persistence_score(real), n_boot=10)["auc_per_person"])
        avg = np.mean([scores[b][f"{t}|{s}"] for s in SEEDS], axis=0)
        r = evaluate(real[t], avg, real.user_id, persistence_score(real), n_boot=200)
        model = pd.Series(r["auc_per_person"])
        base_p = pp(persist[t])
        base_sh = pp(shifted[t])
        base_o = pp(obs[t]) if b else pd.Series(np.nan, index=base_p.index)
        strongest = pd.concat([base_p, base_sh, base_o], axis=1).max(axis=1)
        margin = model - strongest
        summary[(b, t)] = {"AUC": r["auc_median"], "CI95": "%.3f-%.3f" % tuple(r["auc_median_ci95"]),
                           "worst person": model.min(), "persistence AUC": base_p.median(),
                           "shifted persistence AUC": base_sh.median(),
                           "observed-part AUC": base_o.median(),
                           "beats strongest baseline": f"{(margin > 0).sum()}/{margin.notna().sum()}",
                           "smallest margin": margin.min()}
pd.DataFrame(summary).T.round(3)

C:\Users\JM\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\JM\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


AUC         CI95 worst person persistence AUC  \
0  y_f  0.713129  0.676-0.729     0.487118        0.582658   
   y_s  0.686753  0.648-0.716     0.575037        0.542755   
15 y_f   0.84526  0.817-0.880     0.727492        0.636459   
   y_s   0.78585  0.766-0.842     0.692081        0.600871   
30 y_f  0.914232  0.899-0.929     0.840182        0.679836   
   y_s  0.953468  0.923-0.970     0.864055        0.703276   

       shifted persistence AUC observed-part AUC beats strongest baseline  \
0  y_f                0.582658               NaN                    17/17   
   y_s                0.542755               NaN                    16/16   
15 y_f                0.588685          0.949994                     0/17   
   y_s                0.545139          0.762313                    14/16   
30 y_f                0.589482          0.986821                     0/17   
   y_s                0.562213          0.904478                    16/16   

       smallest margin  
0  y_f        0.008949  
   y_s        0.041427  
15 y_f       -0.158901  
   y_s       -0.022685  
30 y_f       -0.100089  
   y_s        0.002477

## Decisión

In [4]:
decision = {}
for b in OFFSETS[1:]:
    for t in TARGETS:
        gain = mean.loc[b, t] - mean.loc[0, t]
        s = summary[(b, t)]
        n_ok, n = map(int, s["beats strongest baseline"].split("/"))
        decision[(b, t)] = {"gain": gain, "all persons beat baselines": n_ok == n,
                            "adopt": gain >= 0.01 and n_ok == n}
pd.DataFrame(decision).T

gain all persons beat baselines  adopt
15 y_f  0.141088                      False  False
   y_s  0.098363                      False  False
30 y_f  0.208608                      False  False
   y_s  0.262509                       True   True

## Guardar

In [5]:
out = {"mean": {str(k): v for k, v in mean.to_dict().items()}, "runs": runs.to_dict("records"),
       "summary": {f"b{b}|{t}": v for (b, t), v in summary.items()},
       "decision": {f"b{b}|{t}": v for (b, t), v in decision.items()}}
Path("data/reports/e4b_label_offset.json").write_text(json.dumps(out, indent=2, default=float))
print("-> data/reports/e4b_label_offset.json")

-> data/reports/e4b_label_offset.json
